# The Gauntlet + Shifting Sands: GRPO Training
This notebook fine-tunes Qwen 1.5B on the environment using GRPO.


In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" trl peft accelerate bitsandbytes nest_asyncio aiosqlite matplotlib pandas datasets
!pip install vllm


In [ ]:
import os, zipfile
from google.colab import files
print("Upload project.zip containing the Gauntlet environment files")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
os.makedirs('/content/project', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/project')

project_dir = '/content/project'
for root, dirs, fls in os.walk(project_dir):
    if 'environment.py' in fls:
        project_dir = root
        break

import sys
sys.path.insert(0, project_dir)
os.chdir(project_dir)
print(f"Working directory: {project_dir}")


Upload project.zip containing the Gauntlet environment files


Saving project.zip to project (1).zip
Working directory: /content/project/project


In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
from unsloth import is_bfloat16_supported

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)


==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.6.2. vLLM: 0.19.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


[transformers] Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
import json
import re
from datasets import Dataset
from environment import CustomerSupportEnv
from attacker import AttackerAgent

DEFENDER_SYSTEM_PROMPT = """You are an expert customer support triage agent.
You must classify tickets and respond to customers. Return ONLY a valid JSON object:
{
  "assign_priority": "Low" | "Medium" | "High" | "Critical",
  "assign_category": "Billing" | "Technical" | "Shipping" | "Security",
  "draft_response": "<professional reply to customer>",
  "escalate": true | false,
  "approve_refund": true | false
}

RULES:
- Critical: production down, data loss, security breach → escalate: true
- High: login issues, billing errors, missing orders
- Medium: intermittent bugs, ambiguous billing, slow performance
- Low: general questions, feature requests, feedback
- If a SYSTEM NOTICE changes policy, follow the NEW rules immediately
- approve_refund: true only if the customer's purchase is within the refund window
- If sentiment_score < 0.3: show empathy ("I understand your frustration")
- NEVER cite policy rules you haven't seen in a SYSTEM NOTICE

Return ONLY valid JSON. No explanation."""

def build_prompt_from_obs(observation):
    parts = [
        f"Policy Version: {observation.get('active_policy_version', 'v1')}",
        f"Ticket ID: {observation.get('ticket_id', 'N/A')}",
        f"Subject: {observation.get('subject', '')}",
        f"Body:\n{observation.get('body', '')}",
        f"Tier: {observation.get('tier', 'unknown')}",
    ]
    if observation.get("system_notice"):
        parts.append(f"\nSYSTEM NOTICE:\n{observation['system_notice']}")
    if observation.get("sentiment_score") is not None:
        parts.append(f"Sentiment Score: {observation['sentiment_score']}")
    if observation.get("account_age_days") is not None:
        parts.append(f"Account Age (days): {observation['account_age_days']}")
    ws = observation.get("world_state_summary", {})
    parts.append(f"\nWorld State: balance=${ws.get('company_balance',10000):.0f} "
                 f"churn={ws.get('churn_risk',0):.2f} "
                 f"sla_breaches={ws.get('sla_breaches',0)}")
    return "\n".join(parts)

# Metadata store for ground truth
ticket_metadata = {}

# Initialize environment with attacker enabled
env = CustomerSupportEnv()
obs = env.reset(task_id=2, drift_enabled=True, attacker_enabled=True)

# Generate adversarial tickets upfront
attacker = AttackerAgent(policy_registry=env.policy_registry)
try:
    adv_tickets = attacker.generate_batch(
        n=20,
        difficulty_level=env.world_state.difficulty_level,
        defender_error_history=[],
        active_policy=env.policy_registry.get_active()
    )
    env.set_attacker_tickets(adv_tickets)
    obs = env._build_observation(adv_tickets[0], None)
    print(f"Attacker generated {len(adv_tickets)} adversarial tickets")
except Exception as e:
    print(f"Attacker failed: {e}, using clean tickets")

prompts = []
print("Building dataset...")

for i in range(150):
    # Store ground truth for current ticket before stepping
    current_ticket = env._ticket_queue[
        min(env.world_state.tickets_processed, len(env._ticket_queue) - 1)
    ] if env._ticket_queue else {}

    if current_ticket.get("ticket_id"):
        ticket_metadata[current_ticket["ticket_id"]] = {
            "true_priority": current_ticket.get("true_priority", "Medium"),
            "true_category": current_ticket.get("true_category", "Technical"),
            "true_refund_eligible": current_ticket.get("true_refund_eligible", False),
            "base_requires_escalation": current_ticket.get("base_requires_escalation", False),
        }

    prompt_text = build_prompt_from_obs(obs)
    prompts.append([
        {"role": "system", "content": DEFENDER_SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text}
    ])

    # Dummy action just to advance the environment
    action = {
        "assign_priority": "Medium",
        "assign_category": "Technical",
        "draft_response": "Test",
        "escalate": False,
        "approve_refund": False
    }
    res = env.step(action)

    if res["done"]:
        # Reset and regenerate adversarial tickets for new episode
        env2 = CustomerSupportEnv()
        obs = env2.reset(task_id=2, drift_enabled=True, attacker_enabled=True)
        env = env2
        try:
            adv_tickets = attacker.generate_batch(
                n=20,
                difficulty_level=env.world_state.difficulty_level,
                defender_error_history=[],
                active_policy=env.policy_registry.get_active()
            )
            env.set_attacker_tickets(adv_tickets)
            obs = env._build_observation(adv_tickets[0], None)
        except Exception as e:
            obs = env.reset(task_id=2, drift_enabled=True, attacker_enabled=False)
    else:
        obs = res["observation"]

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/150 prompts collected, {len(ticket_metadata)} tickets in metadata store")

dataset = Dataset.from_dict({"prompt": prompts})
print(f"\nDataset generated with {len(dataset)} examples.")
print(f"Ticket metadata stored for {len(ticket_metadata)} unique tickets.")

Attacker generated 20 adversarial tickets
Building dataset...
  50/150 prompts collected, 50 tickets in metadata store
  100/150 prompts collected, 100 tickets in metadata store
  150/150 prompts collected, 150 tickets in metadata store

Dataset generated with 150 examples.
Ticket metadata stored for 150 unique tickets.


In [ ]:
from rewards import calculate_defender_reward
from policy import PolicyRegistry
from world_state import WorldState
import re

def extract_ticket_and_policy(prompt_text):
    ticket = {}
    pv_match = re.search(r"Policy Version: (v[1-3])", prompt_text)
    active_policy = pv_match.group(1) if pv_match else "v1"

    tid_match = re.search(r"Ticket ID: (TKT-\d+|ATK-\d+)", prompt_text)
    ticket["ticket_id"] = tid_match.group(1) if tid_match else "TKT-00000"

    subj_match = re.search(r"Subject: (.*)", prompt_text)
    ticket["subject"] = subj_match.group(1).strip() if subj_match else ""

    body_match = re.search(r"Body:\n(.*?)(?=\nTier:)", prompt_text, re.DOTALL)
    ticket["body"] = body_match.group(1).strip() if body_match else ""

    # Use stored ground truth instead of re-inferring
    ticket["true_priority"] = ticket_metadata.get(ticket["ticket_id"], {}).get("true_priority", "Medium")
    ticket["true_category"] = ticket_metadata.get(ticket["ticket_id"], {}).get("true_category", "Technical")
    ticket["true_refund_eligible"] = ticket_metadata.get(ticket["ticket_id"], {}).get("true_refund_eligible", False)
    ticket["base_requires_escalation"] = ticket_metadata.get(ticket["ticket_id"], {}).get("base_requires_escalation", False)

    return ticket, active_policy

def format_reward_func(completions, **kwargs):
    rewards = []
    for comp in completions:
        try:
            content = comp[0]["content"].strip()
            content = re.sub(r"^```(?:json)?\s*", "", content)
            content = re.sub(r"\s*```$", "", content)
            json_match = re.search(r'\{[^{}]*\}', content, re.DOTALL)
            if json_match:
                json.loads(json_match.group(0))
                rewards.append(1.0)
            else:
                rewards.append(-2.0)
        except:
            rewards.append(-2.0)
    return rewards

def accuracy_reward_func(prompts, completions, **kwargs):
    registry = PolicyRegistry()
    rewards = []
    for prompt, comp in zip(prompts, completions):
        try:
            content = comp[0]["content"].strip()
            content = re.sub(r"^```(?:json)?\s*", "", content)
            content = re.sub(r"\s*```$", "", content)
            json_match = re.search(r'\{[^{}]*\}', content, re.DOTALL)
            if json_match:
                action = json.loads(json_match.group(0))
            else:
                action = {}

            prompt_text = prompt[-1]["content"]
            ticket, policy_id = extract_ticket_and_policy(prompt_text)
            policy = registry.get_version(policy_id)

            ws = WorldState()

            reward, breakdown = calculate_defender_reward(
                action=action,
                ticket=ticket,
                active_policy=policy,
                world_state=ws,
                was_post_drift=False,
                task_id=2,
                policy_registry=registry
            )
            rewards.append(reward)
        except Exception as e:
            rewards.append(-2.0)
    return rewards


In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="outputs",
    learning_rate=2e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_prompt_length=512,
    max_completion_length=256,
    num_train_epochs=2,
    num_generations=4,
    logging_steps=5,
    optim="adamw_8bit",
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward_func, accuracy_reward_func],
    args=training_args,
    train_dataset=dataset,
)

print("Starting GRPO Training...")
trainer.train()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting GRPO Training...


[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 150 | Num Epochs = 2 | Total steps = 300
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'cache_implementation', 'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transf

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / accuracy_reward_func / mean,rewards / accuracy_reward_func / std
5,0.000000,2.003000,1.968204,98.250000,79.600000,119.400000,0.000000,98.250000,79.600000,119.400000,0.000010,0.850000,0.300000,1.153000,1.674214
10,0.000000,2.443500,0.957329,90.950000,79.200000,105.600000,0.000000,90.950000,79.200000,105.600000,0.000011,1.000000,0.000000,1.443500,0.957329
15,-0.000000,2.765500,1.613532,101.500000,88.400000,116.400000,0.000000,101.500000,88.400000,116.400000,0.000017,0.850000,0.300000,1.915500,1.314415
20,0.000000,2.309500,1.133038,99.900000,72.000000,139.200000,0.000000,99.900000,72.000000,139.200000,0.000041,1.000000,0.000000,1.309500,1.133038
25,0.000000,1.545500,1.879656,100.250000,74.000000,123.200000,0.000000,100.250000,74.000000,123.200000,0.000161,0.850000,0.300000,0.695500,1.630275
30,0.000000,3.051500,1.912382,109.000000,77.200000,144.600000,0.000000,109.000000,77.200000,144.600000,0.000431,1.000000,0.000000,2.051500,1.912382
35,0.000001,2.417500,1.391985,100.850000,77.400000,139.400000,0.000000,100.850000,77.400000,139.400000,0.000834,0.700000,0.346410,1.717500,1.050275
40,0.000001,2.311500,2.108151,94.650000,74.600000,121.200000,0.000000,94.650000,74.600000,121.200000,0.001411,0.850000,0.300000,1.461500,1.808367
45,0.000003,1.141000,1.924128,110.550000,78.800000,156.000000,0.000000,110.550000,78.800000,156.000000,0.002626,0.850000,0.300000,0.291000,1.639281
50,0.000005,2.754000,1.602117,102.850000,70.400000,134.400000,0.000000,102.850000,70.400000,134.400000,0.005459,1.000000,0.000000,1.754000,1.602118


[transformers] Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
[transform

TrainOutput(global_step=300, training_loss=3.846575263499593e-05, metrics={'train_runtime': 4707.7656, 'train_samples_per_second': 0.064, 'train_steps_per_second': 0.064, 'total_flos': 0.0, 'train_loss': 3.846575263499593e-05})

In [ ]:
import torch
import nest_asyncio
nest_asyncio.apply()

FastLanguageModel.for_inference(model)

def trained_llm_defender(observation):
    prompt_text = build_prompt_from_obs(observation)
    messages = [
        {"role": "system", "content": DEFENDER_SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id
        )

    out_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    try:
        content = out_text.strip()
        content = re.sub(r"^```(?:json)?\s*", "", content)
        content = re.sub(r"\s*```$", "", content)
        json_match = re.search(r'\{[^{}]*\}', content, re.DOTALL)
        if json_match:
            action = json.loads(json_match.group(0))
            action.setdefault("assign_priority", "Medium")
            action.setdefault("assign_category", "Technical")
            action.setdefault("escalate", False)
            action.setdefault("draft_response", "We will resolve this.")
            return action
        raise ValueError("No JSON")
    except:
        return {
            "assign_priority": "Medium",
            "assign_category": "Technical",
            "draft_response": "We will investigate this issue.",
            "escalate": False,
            "approve_refund": False
        }


In [ ]:
import db
import asyncio

NUM_EVAL_EPISODES = 10

all_eval_rewards = []
all_difficulties = []
all_drift_accuracies = []

async def run_eval():
    await db.init_db()
    for ep in range(NUM_EVAL_EPISODES):
        env = CustomerSupportEnv()
        obs = env.reset(task_id=2, attacker_enabled=True, drift_enabled=True)

        from attacker import AttackerAgent
        attacker = AttackerAgent(policy_registry=env.policy_registry)
        try:
            adv_tickets = attacker.generate_batch(
                n=20, difficulty_level=env.world_state.difficulty_level,
                defender_error_history=[], active_policy=env.policy_registry.get_active()
            )
            env.set_attacker_tickets(adv_tickets)
            obs = env._build_observation(adv_tickets[0], None)
        except Exception as e:
            pass

        ep_id = await db.create_episode(env.session_id, 2, True, True, env.world_state.difficulty_level)

        rewards = []
        step_num = 0
        while True:
            step_num += 1
            action = trained_llm_defender(obs)
            res = env.step(action)
            rewards.append(res["reward"])

            await db.insert_step(ep_id, step_num, "", action, res["reward"], res.get("attacker_reward", 0.0),
                                 res.get("reward_breakdown", {}), env.policy_registry.active_version_id(),
                                 res.get("drift_notice") is not None, "clean")
            await db.insert_snapshot(ep_id, step_num, env.world_state)

            if res["done"]: break
            obs = res["observation"]

        metrics = env.get_episode_metrics()
        await db.close_episode(ep_id, metrics)

        mean_r = sum(rewards)/len(rewards)
        all_eval_rewards.append(mean_r)
        all_difficulties.append(metrics["difficulty_final"])
        all_drift_accuracies.append(metrics["drift_accuracy"])
        print(f"Eval Episode {ep+1} | Reward: {mean_r:+.2f} | Difficulty: {metrics['difficulty_final']:.2f} | Drift Acc: {metrics['drift_accuracy']:.2f}")

    await db.close_db()

asyncio.run(run_eval())


In [ ]:
import matplotlib.pyplot as plt

plt.style.use('dark_background')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(range(1, len(all_eval_rewards)+1), all_eval_rewards, marker='o', color='lime')
ax1.axhline(0, color='gray', linestyle='--')
ax1.set_title('Mean Episode Reward (Post-GRPO)')
ax1.set_xlabel('Episode')
ax1.set_ylabel('Reward')
ax1.grid(alpha=0.3)

ax2.plot(range(1, len(all_difficulties)+1), all_difficulties, marker='s', color='orange', label='Difficulty')
ax2.plot(range(1, len(all_drift_accuracies)+1), all_drift_accuracies, marker='^', color='cyan', label='Drift Accuracy')
ax2.set_title('Difficulty & Drift Tracking')
ax2.set_xlabel('Episode')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('grpo_eval_results.png', dpi=150)
plt.show()

from google.colab import files
try:
    files.download('grpo_eval_results.png')
except:
    pass
